In [4]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import optuna
import joblib
import warnings
warnings.filterwarnings("ignore")
import pickle
import pandas as pd
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score

In [5]:
splits = joblib.load("splits/8_windows_all_features_knn.pkl")

In [6]:
splits[0]['X_train_raw'].shape

(1380, 1065)

In [7]:
from sklearn.impute import KNNImputer, SimpleImputer

feature_names = splits[0]["X_train_raw"].columns

for split in splits:

    # imputer = SimpleImputer(strategy="median")
    imputer = KNNImputer(n_neighbors=5, weights="uniform", metric="nan_euclidean")

    split["X_train_raw"] = pd.DataFrame(imputer.fit_transform(split["X_train_raw"]), columns=feature_names)
    split["X_val_raw"] = pd.DataFrame(imputer.transform(split["X_val_raw"]), columns=feature_names)
    split["X_test_raw"] = pd.DataFrame(imputer.transform(split["X_test_raw"]), columns=feature_names)

    vlr_match_id_train = split["X_train_raw"]["vlr_match_id"]
    vlr_match_id_val = split["X_val_raw"]["vlr_match_id"]
    vlr_match_id_test = split["X_test_raw"]["vlr_match_id"]

    split["X_train_raw"] = split["X_train_raw"].loc[:, split["X_train_raw"].columns.str.startswith('delta_')]
    split["X_val_raw"] = split["X_val_raw"].loc[:, split["X_val_raw"].columns.str.startswith('delta_')]
    split["X_test_raw"] = split["X_test_raw"].loc[:, split["X_test_raw"].columns.str.startswith('delta_')]

    split["X_train_raw"]["vlr_match_id"] = vlr_match_id_train
    split["X_val_raw"]["vlr_match_id"] = vlr_match_id_val
    split["X_test_raw"]["vlr_match_id"] = vlr_match_id_test


In [8]:
splits[0]['X_train_raw'].shape

(1380, 355)

In [9]:
def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi

In [10]:
# def make_objective(X_train, y_train, X_val, y_val):
#     def objective(trial):
#         C = trial.suggest_float('C', 1e-3, 100, log=True)
#         l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1)

#         all_val_true, all_val_proba = [], []

#         scaler = StandardScaler()
#         X_train_scaled = scaler.fit_transform(X_train)
#         X_val_scaled   = scaler.transform(X_val)

#         selector = VarianceThreshold(threshold=0.01)
#         X_train_sel = selector.fit_transform(X_train_scaled)
#         X_val_sel   = selector.transform(X_val_scaled)

#         model = LogisticRegression(C=C, l1_ratio=l1_ratio, solver='saga', max_iter=5000)
#         model.fit(X_train_sel, y_train)

#         val_proba = model.predict_proba(X_val_sel)[:, 1]
#         all_val_true.append(y_val); all_val_proba.append(val_proba)
#         logloss = log_loss(all_val_true, all_val_proba)
#         return logloss

#     return objective

In [11]:
from sklearn.model_selection import StratifiedKFold

def make_objective(X_train, y_train, n_inner_folds=5, seed=0):
    def objective(trial):
        C = trial.suggest_float('C', 1e-3, 10, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)

        skf = StratifiedKFold(n_splits=n_inner_folds, shuffle=True, random_state=seed)
        fold_losses = []

        for train_idx, inner_val_idx in skf.split(X_train, y_train):
            X_fold_train, X_fold_val = X_train[train_idx], X_train[inner_val_idx]

            scaler = StandardScaler()
            X_fold_train_scaled = scaler.fit_transform(X_fold_train)
            X_fold_val_scaled = scaler.transform(X_fold_val)

            selector = VarianceThreshold(threshold=0.01)
            X_fold_train_sel = selector.fit_transform(X_fold_train_scaled)
            X_fold_val_sel = selector.transform(X_fold_val_scaled)

            model = LogisticRegression(C=C, l1_ratio=l1_ratio, solver="saga", max_iter=5000)
            model.fit(X_fold_train_sel, y_train[train_idx])
            proba = model.predict_proba(X_fold_val_sel)[:, 1]
            fold_losses.append(log_loss(y_train[inner_val_idx], proba, labels=[0, 1]))

        return np.mean(fold_losses)

    return objective

In [12]:
# features = []
# parameters = []

# for split in splits:
#     X_train, y_train, X_val, y_val = split["X_train_raw"], split["y_train"], split["X_val_raw"], split["y_val"]

#     scaler = StandardScaler()
#     X_train_scaled = scaler.fit_transform(X_train)
#     X_val_scaled   = scaler.transform(X_val)

#     selector = VarianceThreshold(threshold=0.01)
#     X_train_sel = selector.fit_transform(X_train_scaled)
#     X_val_sel   = selector.transform(X_val_scaled)

#     model = LogisticRegression(solver='saga', max_iter=5000)
#     model.fit(X_train_sel, y_train)

#     feature_name_coef = list(zip(X_train.columns, model.coef_[0]))
#     feature_name_coef.sort(key=lambda x: abs(x[1]), reverse=True)
#     features.append([name for name, coef in feature_name_coef[:30]])

In [13]:
features = []

for split in splits:
    X_train, y_train, X_val, y_val = split["X_train_raw"], split["y_train"], split["X_val_raw"], split["y_val"]

    # objective = make_objective(X_train, y_train, X_val, y_val)
    objective = make_objective(X_train.values, y_train.values)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=100, n_jobs=-1)
    params = study.best_params

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel   = selector.transform(X_val_scaled)

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    feature_name_coef = list(zip(X_train.columns, model.coef_[0]))
    feature_name_coef.sort(key=lambda x: abs(x[1]), reverse=True)
    features.append([name for name, coef in feature_name_coef[:30]])

[I 2026-09-10 15:01:54,947] A new study created in memory with name: no-name-93ac959c-281b-4d5a-93b9-598d4f7fd5ba
[I 2026-09-10 15:01:57,145] Trial 10 finished with value: 0.6794841461806914 and parameters: {'C': 0.0027379569747386966, 'l1_ratio': 0.32632800258806227}. Best is trial 10 with value: 0.6794841461806914.
[I 2026-09-10 15:01:58,286] Trial 12 finished with value: 0.5692368399026602 and parameters: {'C': 0.014031212646171659, 'l1_ratio': 0.06927700031698902}. Best is trial 12 with value: 0.5692368399026602.
[I 2026-09-10 15:01:58,756] Trial 0 finished with value: 0.6488886469092723 and parameters: {'C': 0.010430530861113027, 'l1_ratio': 0.5583695315268911}. Best is trial 12 with value: 0.5692368399026602.
[I 2026-09-10 15:01:59,834] Trial 15 finished with value: 0.6589610322596698 and parameters: {'C': 0.012973114642251605, 'l1_ratio': 0.9277414361594682}. Best is trial 12 with value: 0.5692368399026602.
[I 2026-09-10 15:02:00,869] Trial 9 finished with value: 0.6554411987575

In [14]:
scores = []
parameters = []

for split, selected_features in zip(splits, features):
    X_train, y_train, X_val, y_val = split["X_train_raw"], split["y_train"], split["X_val_raw"], split["y_val"]

    X_train = X_train[selected_features]
    X_val = X_val[selected_features]

    # objective = make_objective(X_train, y_train, X_val, y_val)
    objective = make_objective(X_train.values, y_train.values)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=500, n_jobs=-1)
    params = study.best_params
    parameters.append(params)
    print("Best parameters:", params)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel   = selector.transform(X_val_scaled)

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    val_pred = model.predict(X_val_sel)
    scores.append(roc_auc_score(y_val, val_pred))

print(f"Scores: {scores}")
print(f"Parameters: {parameters}")
print(f"Mean roc-auc: {np.mean(scores)}")

[I 2026-09-10 16:47:06,955] A new study created in memory with name: no-name-cebe2477-c52a-46de-aa74-99109b2c299c


[I 2026-09-10 16:47:07,411] Trial 0 finished with value: 0.6481059203548638 and parameters: {'C': 0.8009075758291069, 'l1_ratio': 0.8749186930231899}. Best is trial 0 with value: 0.6481059203548638.
[I 2026-09-10 16:47:07,444] Trial 11 finished with value: 0.6536630878089194 and parameters: {'C': 0.004545390571282353, 'l1_ratio': 0.03806137100531859}. Best is trial 0 with value: 0.6481059203548638.
[I 2026-09-10 16:47:07,452] Trial 1 finished with value: 0.6931119311629498 and parameters: {'C': 0.0014290123579572434, 'l1_ratio': 0.5140990386538604}. Best is trial 0 with value: 0.6481059203548638.
[I 2026-09-10 16:47:07,470] Trial 3 finished with value: 0.6931095649517889 and parameters: {'C': 0.0019407897513168209, 'l1_ratio': 0.3686955611143955}. Best is trial 0 with value: 0.6481059203548638.
[I 2026-09-10 16:47:07,477] Trial 2 finished with value: 0.6649163167805335 and parameters: {'C': 0.008125344180859705, 'l1_ratio': 0.39074730739353836}. Best is trial 0 with value: 0.6481059203

Best parameters: {'C': 0.03235284138708342, 'l1_ratio': 0.00011053604464882766}


[I 2026-09-10 16:47:25,346] Trial 1 finished with value: 0.6928373390755503 and parameters: {'C': 0.0013841341998630534, 'l1_ratio': 0.5678564816892332}. Best is trial 1 with value: 0.6928373390755503.
[I 2026-09-10 16:47:25,372] Trial 5 finished with value: 0.6541492613520374 and parameters: {'C': 5.8342738531589715, 'l1_ratio': 0.8820604762235407}. Best is trial 5 with value: 0.6541492613520374.
[I 2026-09-10 16:47:25,395] Trial 0 finished with value: 0.6540996443426714 and parameters: {'C': 2.3617966962682235, 'l1_ratio': 0.44186075742635456}. Best is trial 0 with value: 0.6540996443426714.
[I 2026-09-10 16:47:25,407] Trial 2 finished with value: 0.6892397595786484 and parameters: {'C': 0.002301508647150573, 'l1_ratio': 0.3217035845670806}. Best is trial 0 with value: 0.6540996443426714.
[I 2026-09-10 16:47:25,410] Trial 3 finished with value: 0.6530169309296598 and parameters: {'C': 0.05817648943415772, 'l1_ratio': 0.633993840946057}. Best is trial 3 with value: 0.6530169309296598.

Best parameters: {'C': 0.016125302529960882, 'l1_ratio': 0.0002552884450807108}


...

Best parameters: {'C': 0.06999262582196647, 'l1_ratio': 0.9631118025051034}


[I 2026-09-10 16:49:07,158] Trial 0 finished with value: 0.6926761920611731 and parameters: {'C': 0.0010811417375907763, 'l1_ratio': 0.30744406138832525}. Best is trial 0 with value: 0.6926761920611731.
[I 2026-09-10 16:49:07,214] Trial 15 finished with value: 0.69267166603813 and parameters: {'C': 0.0020302888722363056, 'l1_ratio': 0.8493653223493264}. Best is trial 15 with value: 0.69267166603813.
[I 2026-09-10 16:49:07,227] Trial 14 finished with value: 0.6926799594740888 and parameters: {'C': 0.0012740044305084568, 'l1_ratio': 0.56467002704357}. Best is trial 15 with value: 0.69267166603813.
[I 2026-09-10 16:49:07,555] Trial 16 finished with value: 0.6894573118812557 and parameters: {'C': 0.003413035149092498, 'l1_ratio': 0.6601172958033694}. Best is trial 16 with value: 0.6894573118812557.
[I 2026-09-10 16:49:07,595] Trial 8 finished with value: 0.6771128910385076 and parameters: {'C': 0.012018805528722126, 'l1_ratio': 0.9339167398819938}. Best is trial 8 with value: 0.67711289103

Best parameters: {'C': 0.08041415106873516, 'l1_ratio': 0.9985816545399101}


[I 2026-09-10 16:49:36,104] Trial 3 finished with value: 0.6926737886916599 and parameters: {'C': 0.0018945756051311372, 'l1_ratio': 0.6786962568425342}. Best is trial 3 with value: 0.6926737886916599.
[I 2026-09-10 16:49:36,193] Trial 4 finished with value: 0.6868874052889163 and parameters: {'C': 0.002500471099705479, 'l1_ratio': 0.3896106806125189}. Best is trial 4 with value: 0.6868874052889163.
[I 2026-09-10 16:49:36,214] Trial 12 finished with value: 0.683436476830532 and parameters: {'C': 0.002316520225437573, 'l1_ratio': 0.2714091733645574}. Best is trial 12 with value: 0.683436476830532.
[I 2026-09-10 16:49:36,290] Trial 5 finished with value: 0.6867177783208841 and parameters: {'C': 0.004976518294649024, 'l1_ratio': 0.8072208460688876}. Best is trial 12 with value: 0.683436476830532.
[I 2026-09-10 16:49:36,363] Trial 9 finished with value: 0.670358613040257 and parameters: {'C': 0.007329899343364633, 'l1_ratio': 0.11249080030501113}. Best is trial 9 with value: 0.670358613040

Best parameters: {'C': 0.06111643405282459, 'l1_ratio': 0.7281286356660254}
Scores: [0.5990280579497524, 0.5605102040816327, 0.6567045000855529, 0.6004759238521836, 0.5792553191489361, 0.5666489346809643, 0.5672507167887861, 0.5764192139737991]
Parameters: [{'C': 0.03235284138708342, 'l1_ratio': 0.00011053604464882766}, {'C': 0.016125302529960882, 'l1_ratio': 0.0002552884450807108}, {'C': 0.24743444755863572, 'l1_ratio': 0.9667385149953265}, {'C': 0.02073176357863719, 'l1_ratio': 0.0003489405574088563}, {'C': 0.06998262079013826, 'l1_ratio': 0.0005731264604019856}, {'C': 0.06999262582196647, 'l1_ratio': 0.9631118025051034}, {'C': 0.08041415106873516, 'l1_ratio': 0.9985816545399101}, {'C': 0.06111643405282459, 'l1_ratio': 0.7281286356660254}]
Mean roc-auc: 0.5882866088202009


Best parameters: {'C': 0.09750681941092068, 'l1_ratio': 0.9995585639101162}
Scores: [0.6201632129103247, 0.5755102040816327, 0.6403924028973934, 0.6188969764837626, 0.6109742441209407, 0.5641991208668141]
Parameters: [{'C': 0.018106815850875393, 'l1_ratio': 8.967149003731838e-05}, {'C': 0.7266420867985378, 'l1_ratio': 0.8886057619829689}, {'C': 0.12596849532139057, 'l1_ratio': 0.8573565142554178}, {'C': 0.014397512918545604, 'l1_ratio': 6.97621339986504e-05}, {'C': 0.015991938036004197, 'l1_ratio': 0.0003961883721978498}, {'C': 0.09750681941092068, 'l1_ratio': 0.9995585639101162}]
Mean roc-auc: 0.6050226935601447

In [17]:
window_results = []
all_val_true, all_val_proba = [], []
all_test_true, all_test_proba = [], []

for i, (split, params, selected_features) in enumerate(zip(splits, parameters, features)):
    X_train, y_train = split["X_train_raw"], split["y_train"]
    X_val, y_val = split["X_val_raw"], split["y_val"]
    X_test, y_test = split["X_test_raw"], split["y_test"]

    X_train = X_train[selected_features]
    X_val = X_val[selected_features]
    X_test = X_test[selected_features]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    X_train_scaled = pd.DataFrame(X_train_scaled, columns=selected_features)
    X_val_scaled = pd.DataFrame(X_val_scaled, columns=selected_features)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=selected_features)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel = selector.transform(X_val_scaled)
    X_test_sel = selector.transform(X_test_scaled)

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    val_proba = model.predict_proba(X_val_sel)[:, 1]
    test_proba = model.predict_proba(X_test_sel)[:, 1]

    window_results.append({
        "window": i,
        "n_val": len(y_val),
        "n_test": len(y_test),
        "val_logloss": log_loss(y_val, val_proba),
        "test_logloss": log_loss(y_test, test_proba),
        "test_acc": accuracy_score(y_test, (test_proba > 0.5).astype(int)),
    })

    all_val_true.append(y_val); all_val_proba.append(val_proba)
    all_test_true.append(y_test); all_test_proba.append(test_proba)

# --- Pool everything: concatenate, don't average ---
# Concatenating before scoring naturally weights each window by its
# sample count. Window 6 (252 matches) counts 252x, Window 2 (32) counts
# 32x — no manual weighting needed, unlike averaging six per-window scores.

pooled_val_true = np.concatenate(all_val_true)
pooled_val_proba = np.concatenate(all_val_proba)
pooled_test_true = np.concatenate(all_test_true)
pooled_test_proba = np.concatenate(all_test_proba)

pooled_test_logloss = log_loss(pooled_test_true, pooled_test_proba)
pooled_test_brier = brier_score_loss(pooled_test_true, pooled_test_proba)
pooled_test_acc = accuracy_score(pooled_test_true, (pooled_test_proba > 0.5).astype(int))

print(f"Pooled test log-loss: {pooled_test_logloss:.4f}")
print(f"Pooled test Brier:    {pooled_test_brier:.4f}")
print(f"Pooled test accuracy: {pooled_test_acc:.4f}  (n={len(pooled_test_true)})")

for r in window_results:
    print(r)


# --- Bootstrap CI, so you can tell if an iteration actually moved the needle ---

def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi


mean_ll, lo, hi = bootstrap_logloss_ci(pooled_test_true, pooled_test_proba)
print(f"Pooled log-loss: {mean_ll:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Pooled test log-loss: 0.6849
Pooled test Brier:    0.2446
Pooled test accuracy: 0.6128  (n=656)
{'window': 0, 'n_val': 297, 'n_test': 102, 'val_logloss': 0.6160098953902314, 'test_logloss': 0.7929781539838279, 'test_acc': 0.5588235294117647}
{'window': 1, 'n_val': 198, 'n_test': 48, 'val_logloss': 0.7179088768504266, 'test_logloss': 0.6573506830226618, 'test_acc': 0.5625}
{'window': 2, 'n_val': 375, 'n_test': 72, 'val_logloss': 0.629971975384221, 'test_logloss': 0.6798559962275534, 'test_acc': 0.5833333333333334}
{'window': 3, 'n_val': 378, 'n_test': 102, 'val_logloss': 0.6658972156711366, 'test_logloss': 0.6430747700385498, 'test_acc': 0.6274509803921569}
{'window': 4, 'n_val': 378, 'n_test': 72, 'val_logloss': 0.659403409527448, 'test_logloss': 0.6229529444217415, 'test_acc': 0.7916666666666666}
{'window': 5, 'n_val': 378, 'n_test': 56, 'val_logloss': 0.6831301190908927, 'test_logloss': 0.6771481048969203, 'test_acc': 0.5535714285714286}
{'window': 6, 'n_val': 434, 'n_test': 24, 'val

Pooled test log-loss: 0.6991
Pooled test Brier:    0.2505
Pooled test accuracy: 0.6305  (n=452)
{'window': 0, 'n_val': 297, 'n_test': 102, 'val_logloss': 0.6110449366927905, 'test_logloss': 0.7904254984225982, 'test_acc': 0.5588235294117647}
{'window': 1, 'n_val': 198, 'n_test': 48, 'val_logloss': 0.737514643554363, 'test_logloss': 0.747652112307672, 'test_acc': 0.4583333333333333}
{'window': 2, 'n_val': 375, 'n_test': 72, 'val_logloss': 0.6548906787828987, 'test_logloss': 0.7226250820050232, 'test_acc': 0.7083333333333334}
{'window': 3, 'n_val': 378, 'n_test': 102, 'val_logloss': 0.6519088207608126, 'test_logloss': 0.6378737833967578, 'test_acc': 0.6764705882352942}
{'window': 4, 'n_val': 378, 'n_test': 72, 'val_logloss': 0.651855945000742, 'test_logloss': 0.6233010739981247, 'test_acc': 0.75}
{'window': 5, 'n_val': 378, 'n_test': 56, 'val_logloss': 0.6876066836409673, 'test_logloss': 0.6700529918128618, 'test_acc': 0.5714285714285714}
Pooled log-loss: 0.6993  95% CI [0.6632, 0.7360]

In [18]:
parameters

[{'C': 0.03235284138708342, 'l1_ratio': 0.00011053604464882766},
 {'C': 0.016125302529960882, 'l1_ratio': 0.0002552884450807108},
 {'C': 0.24743444755863572, 'l1_ratio': 0.9667385149953265},
 {'C': 0.02073176357863719, 'l1_ratio': 0.0003489405574088563},
 {'C': 0.06998262079013826, 'l1_ratio': 0.0005731264604019856},
 {'C': 0.06999262582196647, 'l1_ratio': 0.9631118025051034},
 {'C': 0.08041415106873516, 'l1_ratio': 0.9985816545399101},
 {'C': 0.06111643405282459, 'l1_ratio': 0.7281286356660254}]

In [19]:
pd.DataFrame(model.coef_[0], index=selector.get_feature_names_out(), columns=["Coefficient"]).sort_values(by="Coefficient", key=abs, ascending=False)


,Coefficient
delta_match_patch_win_rate,0.189017
delta_match_patch_win_rate_4,-0.180085
delta_match_score_delta_sum_4,0.168576
delta_map_specific_score_delta_avg_4,0.136283
delta_player_map_stats_specific_clutch_1v1_variance_log1p,0.122370
delta_player_map_stats_specific_average_damage_per_round_median_2,0.110320
delta_match_patch_played_ratio,-0.110169
delta_player_map_stats_specific_clutch_1v3_median_6,0.105740
delta_player_map_stats_specific_three_kills_median,0.087448
delta_player_map_stats_specific_average_combat_score_variance_log1p_2,0.061605


In [20]:
np.array(selected_features)[selector.get_support()]

array(['delta_player_map_stats_specific_clutch_1v4_median_2',
       'delta_player_map_stats_specific_four_kills_median',
       'delta_player_map_stats_specific_average_damage_per_round_median_2',
       'delta_player_map_stats_specific_clutch_1v1_median_6',
       'delta_map_specific_score_delta_sum_2',
       'delta_match_patch_win_rate_4',
       'delta_map_specific_score_delta_sum_4',
       'delta_player_map_stats_specific_clutch_1v3_median_4',
       'delta_player_map_stats_specific_average_combat_score_variance_log1p_2',
       'delta_player_map_stats_specific_clutch_1v4_median_4',
       'delta_match_patch_played',
       'delta_player_map_stats_specific_three_kills_median',
       'delta_player_map_stats_specific_deaths_median_2',
       'delta_map_specific_score_delta_avg_4',
       'delta_player_map_stats_specific_first_kills_variance_log1p',
       'delta_match_patch_played_ratio',
       'delta_match_patch_score_delta_avg_6',
       'delta_match_patch_win_rate',
       'd

In [21]:
split.keys()

dict_keys(['X_train_raw', 'y_train', 'X_val_raw', 'y_val', 'X_test_raw', 'y_test'])

In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score

window_results = []
all_val_true, all_val_proba = [], []
all_test_true, all_test_proba = [], []

match_rows = []      # Layer 1: one row per match (val set)
feature_rows = []    # Layer 2: one row per match x feature (val set)

for i, (split, params, selected_features) in enumerate(zip(splits, parameters, features)):
    X_train, y_train = split["X_train_raw"], split["y_train"]
    X_val, y_val = split["X_val_raw"], split["y_val"]
    X_test, y_test = split["X_test_raw"], split["y_test"]

    X_train = X_train[selected_features]
    X_val = X_val[selected_features]
    X_test = X_test[selected_features]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    X_train_scaled = pd.DataFrame(X_train_scaled, columns=selected_features)
    X_val_scaled = pd.DataFrame(X_val_scaled, columns=selected_features)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=selected_features)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel = selector.transform(X_val_scaled)
    X_test_sel = selector.transform(X_test_scaled)

    # names of columns that survived VarianceThreshold, in order
    surviving_features = np.array(selected_features)[selector.get_support()]

    model = LogisticRegression(C=params['C'], l1_ratio=params['l1_ratio'], solver='saga', max_iter=5000)
    model.fit(X_train_sel, y_train)

    val_proba = model.predict_proba(X_val_sel)[:, 1]
    test_proba = model.predict_proba(X_test_sel)[:, 1]

    window_results.append({
        "window": i,
        "n_val": len(y_val),
        "n_test": len(y_test),
        "val_logloss": log_loss(y_val, val_proba),
        "test_logloss": log_loss(y_test, test_proba),
        "test_acc": accuracy_score(y_test, (test_proba > 0.5).astype(int)),
    })

    all_val_true.append(y_val); all_val_proba.append(val_proba)
    all_test_true.append(y_test); all_test_proba.append(test_proba)

    # ---------- Layer 1: match-level results (VAL) ----------
    vlr_match_ids = np.asarray(split['X_val_raw']["vlr_match_id"])  # adjust key to your actual field
    # match_ids = split.get("match_id_val", np.arange(len(y_val)))  # adjust key to your actual field
    y_val_arr = np.asarray(y_val)
    pred_label = (val_proba > 0.5).astype(int)
    eps = 1e-9
    ll_contrib = -(y_val_arr * np.log(np.clip(val_proba, eps, 1 - eps)) +
                   (1 - y_val_arr) * np.log(np.clip(1 - val_proba, eps, 1 - eps)))

    window_match_df = pd.DataFrame({
        "vlr_match_id": vlr_match_ids,
        "window": i,
        "y_true": y_val_arr,
        "y_pred_proba": val_proba,
        "pred_label": pred_label,
        "correct": pred_label == y_val_arr,
        "confidence": np.abs(val_proba - 0.5) * 2,
        "log_loss_contribution": ll_contrib,
        "p_favorite": np.maximum(val_proba, 1 - val_proba),
    })
    window_match_df["favorite_won"] = np.where(
        val_proba > 0.5, window_match_df["y_true"] == 1, window_match_df["y_true"] == 0
    )
    # join metadata here if it lives elsewhere, e.g.:
    # window_match_df = window_match_df.merge(metadata_df, on="match_id", how="left")
    match_rows.append(window_match_df)

    # ---------- Layer 2: feature-level SHAP (VAL) ----------
    # Manual linear-SHAP-style attribution: coef * (x - train_mean), matching
    # your existing manual approach from the deployed SHAP implementation.
    train_mean = X_train_sel.mean(axis=0)
    coefs = model.coef_[0]
    shap_matrix = (np.asarray(X_val_sel) - train_mean) * coefs  # shape (n_val, n_features)

    n_val_rows, n_feats = shap_matrix.shape
    window_feature_df = pd.DataFrame({
        "vlr_match_id": np.repeat(vlr_match_ids, n_feats),
        "window": i,
        "feature_name": np.tile(surviving_features, n_val_rows),
        "feature_value_raw": np.asarray(X_val[surviving_features]).flatten(),
        "feature_value_scaled": np.asarray(X_val_sel).flatten(),
        "shap_value": shap_matrix.flatten(),
    })
    feature_rows.append(window_feature_df)

# ---------- Assemble the two output tables ----------
results_df = pd.concat(match_rows, ignore_index=True)
shap_df = pd.concat(feature_rows, ignore_index=True)

# --- Pool test metrics: unchanged, still the integrity check ---
pooled_val_true = np.concatenate(all_val_true)
pooled_val_proba = np.concatenate(all_val_proba)
pooled_test_true = np.concatenate(all_test_true)
pooled_test_proba = np.concatenate(all_test_proba)

pooled_test_logloss = log_loss(pooled_test_true, pooled_test_proba)
pooled_test_brier = brier_score_loss(pooled_test_true, pooled_test_proba)
pooled_test_acc = accuracy_score(pooled_test_true, (pooled_test_proba > 0.5).astype(int))

print(f"Pooled test log-loss: {pooled_test_logloss:.4f}")
print(f"Pooled test Brier:    {pooled_test_brier:.4f}")
print(f"Pooled test accuracy: {pooled_test_acc:.4f}  (n={len(pooled_test_true)})")

for r in window_results:
    print(r)



Pooled test log-loss: 0.6849
Pooled test Brier:    0.2446
Pooled test accuracy: 0.6128  (n=656)
{'window': 0, 'n_val': 297, 'n_test': 102, 'val_logloss': 0.6160102349147357, 'test_logloss': 0.7929828396414882, 'test_acc': 0.5588235294117647}
{'window': 1, 'n_val': 198, 'n_test': 48, 'val_logloss': 0.7179077884116832, 'test_logloss': 0.6573511065853541, 'test_acc': 0.5625}
{'window': 2, 'n_val': 375, 'n_test': 72, 'val_logloss': 0.6299707067282205, 'test_logloss': 0.6798618127818842, 'test_acc': 0.5833333333333334}
{'window': 3, 'n_val': 378, 'n_test': 102, 'val_logloss': 0.6658955900280408, 'test_logloss': 0.643067804921362, 'test_acc': 0.6274509803921569}
{'window': 4, 'n_val': 378, 'n_test': 72, 'val_logloss': 0.6593989280981094, 'test_logloss': 0.6229479237048646, 'test_acc': 0.7916666666666666}
{'window': 5, 'n_val': 378, 'n_test': 56, 'val_logloss': 0.683129475788515, 'test_logloss': 0.6771465448455665, 'test_acc': 0.5535714285714286}
{'window': 6, 'n_val': 434, 'n_test': 24, 'val

In [40]:
joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(selector, "selector.pkl")
joblib.dump(imputer, "imputer.pkl")

X_bg = X_train.reindex(columns=selector.feature_names_in_)
# X_bg = imputer.transform(X_bg)
X_bg = scaler.transform(X_bg)
X_bg = selector.transform(X_bg)

joblib.dump(X_bg.mean(axis=0), "background_mean.npy")
joblib.dump(X_bg.std(axis=0), "background_std.npy")

['background_std.npy']

In [24]:
results_df

,vlr_match_id,window,y_true,y_pred_proba,pred_label,correct,confidence,log_loss_contribution,p_favorite,favorite_won
0,314340.0,0,1,0.677665,1,True,0.355330,0.389102,0.677665,True
1,314341.0,0,1,0.814137,1,True,0.628273,0.205627,0.814137,True
2,314340.0,0,0,0.328668,0,True,0.342664,0.398492,0.671332,True
3,314341.0,0,0,0.190268,0,True,0.619464,0.211052,0.809732,True
4,314340.0,0,1,0.677665,1,True,0.355330,0.389102,0.677665,True
...,...,...,...,...,...,...,...,...,...,...
2891,708424.0,7,1,0.475514,0,False,0.048971,0.743358,0.524486,False
2892,708426.0,7,1,0.570627,1,True,0.141254,0.561019,0.570627,True
2893,708426.0,7,0,0.406876,0,True,0.186247,0.522353,0.593124,True
2894,708427.0,7,0,0.426789,0,True,0.146421,0.556502,0.573211,True


### Calibration by confidence bucket — is the model's stated confidence honest?

A well-calibrated model's 0.7-0.8 bucket should see the favorite win ~70-80% of the time. Systematic gaps (e.g. that bucket only wins 60%) tell you the model is overconfident in a specific range — that's a concrete, testable failure mode, not vague "accuracy could be better."

In [25]:
results_df["conf_bucket"] = pd.cut(results_df["p_favorite"], bins=[0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
calib = results_df.groupby("conf_bucket").agg(
    n=("favorite_won", "size"),
    empirical_win_rate=("favorite_won", "mean"),
)
print(calib)

                n  empirical_win_rate
conf_bucket                          
(0.5, 0.6]   1639            0.542404
(0.6, 0.7]    842            0.602138
(0.7, 0.8]    305            0.727869
(0.8, 0.9]    104            0.798077
(0.9, 1.0]      6            1.000000


### Confident-and-wrong matches — the highest-value failure mode to inspect individually

This is your shortlist for manually pulling up the actual match pages — if several cluster around a shared trait (recent roster change, a specific region, a big seed gap), that's a feature hypothesis forming directly from evidence.

In [26]:
confident_wrong = results_df[(results_df["confidence"] > 0.5) & (~results_df["correct"])].sort_values(
    "log_loss_contribution", ascending=False
)
confident_wrong.head(20)

,vlr_match_id,window,y_true,y_pred_proba,pred_label,correct,confidence,log_loss_contribution,p_favorite,favorite_won,conf_bucket
528,458809.0,2,1,0.102646,0,False,0.794708,2.276467,0.897354,False,"(0.8, 0.9]"
526,458809.0,2,1,0.102646,0,False,0.794708,2.276467,0.897354,False,"(0.8, 0.9]"
525,458809.0,2,0,0.887500,1,False,0.775000,2.184802,0.887500,False,"(0.8, 0.9]"
438,430531.0,1,1,0.114354,0,False,0.771292,2.168458,0.885646,False,"(0.8, 0.9]"
134,353189.0,0,0,0.883777,1,False,0.767554,2.152246,0.883777,False,"(0.8, 0.9]"
441,430531.0,1,0,0.882029,1,False,0.764057,2.137313,0.882029,False,"(0.8, 0.9]"
433,430531.0,1,0,0.882029,1,False,0.764057,2.137313,0.882029,False,"(0.8, 0.9]"
137,353189.0,0,1,0.119219,0,False,0.761562,2.126795,0.880781,False,"(0.8, 0.9]"
132,353189.0,0,1,0.119219,0,False,0.761562,2.126795,0.880781,False,"(0.8, 0.9]"
572,459527.0,2,1,0.146892,0,False,0.706217,1.918061,0.853108,False,"(0.8, 0.9]"


In [27]:
for i, vlr_match in enumerate(confident_wrong["vlr_match_id"].unique()):
    print(f"{i + 1}: https://www.vlr.gg/{int(vlr_match)}")

1: https://www.vlr.gg/458809
2: https://www.vlr.gg/430531
3: https://www.vlr.gg/353189
4: https://www.vlr.gg/459527
5: https://www.vlr.gg/353198
6: https://www.vlr.gg/458824
7: https://www.vlr.gg/429385
8: https://www.vlr.gg/473844
9: https://www.vlr.gg/473843
10: https://www.vlr.gg/353179
11: https://www.vlr.gg/353163
12: https://www.vlr.gg/314362
13: https://www.vlr.gg/314350
14: https://www.vlr.gg/365317


### Does performance drift across windows (temporal check)?

A steady trend (getting worse in later windows) suggests meta/roster drift outpacing your features' freshness; a flat pattern says the ceiling is stable over time, not eroding.

In [28]:
window_summary = results_df.groupby("window").agg(
    n=("correct", "size"),
    acc=("correct", "mean"),
    mean_logloss=("log_loss_contribution", "mean"),
)
window_summary

,n,acc,mean_logloss
window,,,
0,297,0.602694,0.616010
1,198,0.560606,0.717908
2,375,0.658667,0.629971
3,378,0.600529,0.665896
4,378,0.579365,0.659399
5,378,0.566138,0.683129
6,434,0.566820,0.682404
7,458,0.576419,0.679863


### SHAP: correct vs. incorrect predictions — which features behave differently on failures

Features with a high ratio are pulling extra weight specifically on matches the model gets wrong — worth asking why. Keep n_windows_selected visible since a feature that only appeared in one window's top-30 has a much smaller, less trustworthy sample behind its number.

In [29]:
merged = shap_df.merge(results_df[["vlr_match_id", "window", "correct", "y_true"]], on=["vlr_match_id", "window"])

shap_compare = merged.groupby(["feature_name", "correct"])["shap_value"].apply(
    lambda s: s.abs().mean()
).unstack("correct").rename(columns={True: "mean_abs_shap_correct", False: "mean_abs_shap_wrong"})

shap_compare["n_windows_selected"] = merged.groupby("feature_name")["window"].nunique()
shap_compare["ratio_wrong_to_correct"] = shap_compare["mean_abs_shap_wrong"] / shap_compare["mean_abs_shap_correct"]
shap_compare.sort_values("ratio_wrong_to_correct", ascending=False)

correct,mean_abs_shap_wrong,mean_abs_shap_correct,n_windows_selected,ratio_wrong_to_correct
feature_name,,,,
delta_player_map_stats_specific_five_kills_variance_log1p_6,0.016663,0.010112,2,1.647915
delta_map_patch_score_delta_avg_2,0.037272,0.028761,1,1.295913
delta_player_map_stats_specific_defuses_median,0.014538,0.011359,4,1.279865
delta_player_map_stats_specific_deaths_variance_log1p_6,0.029649,0.023294,3,1.272824
delta_player_map_stats_specific_clutch_1v1_median_6,0.000135,0.000108,3,1.252121
...,...,...,...,...
delta_player_map_stats_specific_clutch_1v4_median_2,0.000079,0.000224,3,0.354556
delta_player_map_stats_specific_plants_median,0.006259,0.047827,1,0.130877
delta_player_map_stats_specific_clutch_1v4_median_4,0.000000,0.000000,3,NaN


In [30]:
shap_compare.describe()

correct,mean_abs_shap_wrong,mean_abs_shap_correct,n_windows_selected,ratio_wrong_to_correct
count,70.000000,70.000000,70.000000,67.000000
mean,0.054036,0.058277,2.714286,0.968030
std,0.063899,0.072111,1.858337,0.223738
min,0.000000,0.000000,1.000000,0.130877
25%,0.012720,0.012053,1.000000,0.840365
50%,0.038466,0.039432,2.000000,0.990151
75%,0.057911,0.060628,3.000000,1.077200
max,0.271051,0.349219,8.000000,1.647915


### Does a specific feature actually vary in the slice you think it explains?

If feature_value_raw is nearly constant across this slice, the feature isn't distinguishing these failures regardless of its SHAP weight elsewhere — it's the direct test from the "I think I already have this feature" discussion.

In [31]:
feature_to_check = "delta_map_specific_score_delta_avg_4"  # example
bad_slice_ids = confident_wrong["vlr_match_id"]

check = shap_df[
    (shap_df["feature_name"] == feature_to_check) & (shap_df["vlr_match_id"].isin(bad_slice_ids))
]
check[["vlr_match_id", "feature_value_raw", "shap_value"]].describe()

,vlr_match_id,feature_value_raw,shap_value
count,0.0,0.0,0.0
mean,NaN,NaN,NaN
std,NaN,NaN,NaN
min,NaN,NaN,NaN
25%,NaN,NaN,NaN
50%,NaN,NaN,NaN
75%,NaN,NaN,NaN
max,NaN,NaN,NaN


### Direction-flip check

If mean SHAP is positive-and-favors-the-actual-winner on correct predictions but positive-and-favors-the-loser on incorrect ones, that's evidence of a confound the feature isn't disentangling (e.g. it correlates with team strength most of the time, but misfires under some condition you haven't captured yet — which is your next hypothesis to chase).

In [32]:
sign_check = merged[merged["feature_name"] == feature_to_check].groupby("correct")["shap_value"].mean()
print(sign_check)

correct
False    0.002651
True     0.006301
Name: shap_value, dtype: float64


In [33]:
flip_check = merged[merged["feature_name"] == feature_to_check].groupby("correct")["shap_favored_winner"].agg(["mean", "size"])
print(flip_check)

KeyError: 'Column not found: shap_favored_winner'

In [ ]:
feature_idx = list(surviving_features).index(feature_to_check)
print(model.coef_[0][feature_idx])

0.13923271969230527


In [ ]:
raw_vals = shap_df[shap_df["feature_name"] == feature_to_check]["feature_value_raw"]
print(raw_vals.skew())
print(raw_vals.describe())
print((raw_vals < raw_vals.mean()).mean())  # fraction below the mean

0.011399511307678412
count    753.000000
mean      -0.146746
std        4.359908
min      -13.000000
25%       -3.000000
50%        0.000000
75%        2.750000
max       13.000000
Name: feature_value_raw, dtype: float64
0.4953519256308101
